In [43]:
!pip install -q gradio
!pip install -q scikit-learn
!pip install -q joblib
!pip install -q pandas 
!pip install -q numpy

In [76]:
from __future__ import annotations

from collections.abc import Iterable

from gradio.themes.base import Base
from gradio.themes.utils import colors, fonts, sizes


class Ocean(Base):
    def __init__(
        self,
        *,
        primary_hue: colors.Color | str = colors.slate,
        secondary_hue: colors.Color | str = colors.sky,
        neutral_hue: colors.Color | str = colors.slate,
        spacing_size: sizes.Size | str = sizes.spacing_md,
        radius_size: sizes.Size | str = sizes.radius_xxl,
        text_size: sizes.Size | str = sizes.text_md,
        font: fonts.Font | str | Iterable[fonts.Font | str] = (
            fonts.GoogleFont("IBM Plex Sans"),
            "ui-sans-serif",
            "system-ui",
            "sans-serif",
        ),
        font_mono: fonts.Font | str | Iterable[fonts.Font | str] = (
            fonts.GoogleFont("IBM Plex Mono"),
            "ui-monospace",
            "Consolas",
            "monospace",
        ),
    ):
        super().__init__(
            primary_hue=primary_hue,
            secondary_hue=secondary_hue,
            neutral_hue=neutral_hue,
            spacing_size=spacing_size,
            radius_size=radius_size,
            text_size=text_size,
            font=font,
            font_mono=font_mono,
        )
        self.name = "ocean"
        super().set(
            body_background_fill="*primary_950",        # True Black Background
            body_text_color="#FFFFFF",            # True White Text
            block_background_fill="*primary_900",      # Slightly lighter black for containers
            block_label_text_color="#FFFFFF",
            input_background_fill="*primary_800",
            
            button_border_width="0px",
            checkbox_label_border_width="1px",
            button_transform_hover="scale(1.02)",
            button_transition="all 0.1s ease-in-out",
            slider_color="*secondary_400",
            button_primary_background_fill="linear-gradient(120deg, *secondary_500 0%, *primary_300 60%, *primary_400 100%)",
            button_primary_background_fill_hover="linear-gradient(120deg, *secondary_400 0%, *primary_300 60%, *primary_300 100%)",
            button_primary_text_color="*button_secondary_text_color",
            button_secondary_background_fill="linear-gradient(120deg, *neutral_300 0%, *neutral_100 60%, *neutral_200 100%)",
            button_secondary_background_fill_hover="linear-gradient(120deg, *neutral_200 0%, *neutral_100 60%, *neutral_100 100%)",
            checkbox_label_background_fill_selected="linear-gradient(120deg, *primary_400 0%, *primary_300 60%, *primary_400 100%)",
            checkbox_label_border_color_selected="*primary_400",
            checkbox_background_color_selected="*primary_400",
            checkbox_label_text_color_selected="*button_secondary_text_color",
            slider_color_dark="*secondary_500",
            button_primary_background_fill_dark="linear-gradient(120deg, *secondary_600 0%, *primary_500 60%, *primary_600 100%)",
            button_primary_background_fill_hover_dark="linear-gradient(120deg, *secondary_500 0%, *primary_500 60%, *primary_500 100%)",
            button_primary_text_color_dark="*button_secondary_text_color",
            button_secondary_background_fill_dark="linear-gradient(120deg, *neutral_700 0%, *neutral_600 60%, *neutral_700 100%)",
            button_secondary_background_fill_hover_dark="linear-gradient(120deg, *neutral_600 0%, *neutral_600 60%, *neutral_700 100%)",
            checkbox_label_background_fill_selected_dark="linear-gradient(120deg, *primary_600 0%, *primary_500 60%, *primary_600 100%)",
            checkbox_label_border_color_selected_dark="*primary_600",
            checkbox_background_color_selected_dark="*primary_600",
            checkbox_label_text_color_selected_dark="*button_secondary_text_color",
            block_shadow="*shadow_drop_lg",
            button_secondary_shadow_hover="*shadow_drop_lg",
            button_primary_shadow_hover="0 1px 3px 0 *primary_200, 0 1px 2px -1px *primary_200",
            button_secondary_shadow_dark="none",
            button_primary_shadow_dark="none",
        )

In [77]:
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RepeatedKFold
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score

file_path = 'Concrete_Data_simplified.csv'
data = pd.read_csv(file_path)

X = data[['Cement ','Blast Furnace Slag ','Fly Ash ','Water ','Superplasticizer','Coarse Aggregate','Fine Aggregate','Age']]
# Cement ,Blast Furnace Slag ,Fly Ash ,Water ,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Concrete compressive strength
y = data['Concrete compressive strength']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rkf = RepeatedKFold(n_splits=5,n_repeats=5,random_state=42)

# using linear regression
lr_model = LinearRegression()
lr_model.fit(X=X_train,y=y_train)

lr_scores = cross_val_score(estimator=lr_model,X=X_train,y=y_train,cv=rkf,scoring='neg_mean_absolute_error')

print("----LINEAR REGRESSION----")
print(f"Negative MAE scores: {lr_scores}")
print(f"Mean cross-validation: {np.mean(lr_scores):.4f}")
print(f"Standard deviation: {np.std(lr_scores):.4f}")

# using random forest
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X=X_train,y=y_train)

rf_scores = cross_val_score(estimator=rf_regressor,X=X_train,y=y_train,cv=rkf,scoring='neg_mean_absolute_error',n_jobs=-1)

print("----RANDOM FOREST----")
print(f"Negative MAE scores: {rf_scores}")
print(f"Mean cross-validation: {np.mean(rf_scores):.4f}")
print(f"Standard deviation: {np.std(rf_scores):.4f}\n")

# prediction results
y_pred_rf = rf_regressor.predict(X_test)
test_mae_rf = mean_absolute_error(y_test, y_pred_rf)
accuracy_rf = rf_regressor.score(X_test, y_test) * 100

print(f"Test MAE (RF): {test_mae_rf:.4f}")
print(f"Test Accuracy (RF): {accuracy_rf:.2f}%")

y_pred_lr = lr_model.predict(X_test)
test_mae_lr = mean_absolute_error(y_test, y_pred_lr)
accuracy_lf = lr_model.score(X_test, y_test) * 100

print(f"Test MAE (LR): {test_mae_lr:.4f}")
print(f"Test Accuracy (LR): {accuracy_lf:.2f}%")

joblib.dump(rf_regressor, 'rf_model.joblib')
content = {"MAE":test_mae_rf, "Accuracy":accuracy_rf}
file_path = "data.json"
try:
    with open(file_path, "w") as json_file:
        json.dump(content, json_file)
    print(f"Successfully created {file_path}")
except IOError as e:
    print(f"Error creating file: {e}")

----LINEAR REGRESSION----
Negative MAE scores: [-8.59468472 -8.16626241 -8.97924297 -8.57543611 -7.79246166 -8.20063685
 -8.31495927 -8.62194636 -8.19822736 -8.89095727 -8.2641171  -8.17348477
 -8.56039265 -8.48931401 -8.69609518 -8.36632343 -8.0853727  -9.0394019
 -8.51601324 -8.27644583 -8.88961754 -8.03579759 -8.58116449 -8.36384936
 -8.47712921]
Mean cross-validation: -8.4460
Standard deviation: 0.3029
----RANDOM FOREST----
Negative MAE scores: [-3.46696563 -4.05680377 -3.87451224 -3.77730118 -3.44161267 -4.13064565
 -3.10731499 -3.75342397 -3.83888218 -3.33875748 -4.2001808  -3.91500097
 -3.09685843 -3.83339575 -3.36641002 -3.94458004 -3.64763106 -4.22398079
 -3.6735098  -3.30217245 -3.35461159 -3.56965922 -4.47484704 -3.34872924
 -3.85387297]
Mean cross-validation: -3.7037
Standard deviation: 0.3546

Test MAE (RF): 3.7363
Test Accuracy (RF): 88.41%
Test MAE (LR): 7.7456
Test Accuracy (LR): 62.76%
Successfully created data.json


In [78]:
import gradio as gr
from joblib import load
import pandas as pd
import json

MAE_CV = 0

try:
    with open('data.json', 'r') as file:
        # Load the JSON data into a Python dictionary
        data = json.load(file)

    # Now you can work with the data as a normal Python dictionary
    MAE_CV = data["MAE"]

except FileNotFoundError:
    print("Error: The file 'data.json' was not found.")
except json.JSONDecodeError:
    print("Error: Failed to decode JSON from the file. Check if the JSON is valid.")

# These are the constants
K = 2         
MARGIN = K * MAE_CV

rf_model = load('rf_model.joblib')
    
# Prediction + Decision Logic
def decision_engine(cement: float, slag: float, ash: float, water: float, plasticizer: float, coarse: float, fine: float, age: float, target_strength: float):

    input_data = pd.DataFrame([[cement, slag, ash, water, plasticizer, coarse, fine, age]],
        columns=['Cement ','Blast Furnace Slag ','Fly Ash ','Water ','Superplasticizer','Coarse Aggregate','Fine Aggregate','Age'])
    prediction = rf_model.predict(input_data)

    # Logic part for the decision engine
    if (prediction - MARGIN) >= target_strength:
        status = "✅ PASS"
        note = "Safe to proceed: The mix design is reliably above the target."
    elif (prediction + MARGIN) <= target_strength:
        status = "❌ FAIL"
        note = "Unsafe: The mix design is reliably below the target."
    else:
        status = "⚠️ BORDERLINE"
        note = "Uncertain: Requires manual lab verification before use."

    # RETURN TWO SEPARATE THINGS:
    # 1st: The Strength Info
    strength_info = f"{np.round(prediction, 4)} MPa (Margin: ±{MARGIN})"
    # 2nd: The Decision Info
    decision_info = f"Status: {status}\n\nAdvisory: {note}\n\nNote: Decision support only; not a substitute for standard lab testing."
    
    return strength_info, decision_info

# 3. the ui
demo = gr.Interface(
    fn=decision_engine,
    inputs=[
        gr.Slider(0, 1500, label="Cement (kg)"),
        gr.Slider(0, 1500, label="Blast Furnace Slag (kg)"),
        gr.Slider(0, 1500, label="Fly Ash (kg)"),
        gr.Slider(0, 1500, label="Water (kg)"),
        gr.Slider(0, 1500, label="Superplasticizer (kg)"),
        gr.Slider(0, 1500, label="Coarse Aggregate (kg)"),
        gr.Slider(0, 1500, label="Fine Aggregate (kg)"),
        gr.Slider(0, 1500, label="Age (Days)"),
        gr.Number(label="Required Target Strength f'c (MPa)", value=10)
    ],
    outputs=[
        gr.Textbox(label="AI Predicted Compressive Strength", lines=2),
        gr.Textbox(label="Decision Support Output", lines=8),
        #gr.Textbox(label="Decision Support Output", lines=8)
    ],
    title="Concrete Compressive Strength Prediction and Decision-Support System Using Supervised Machine Learning",
    theme=Ocean(primary_hue="slate", secondary_hue="blue", neutral_hue="slate"),
    live=True,
    allow_flagging="never"
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aed0dd4baa085eaa1c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
